In [1]:
import sys
print(sys.executable)


d:\Projects\Kaggle projects\venv\Scripts\python.exe


In [12]:
import vpplib
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [11]:
# Create an environment (adjust parameters as needed)
env = vpplib.Environment(timebase=15, timezone='Europe/Berlin')
env

In [15]:
weather_data = pd.DataFrame({
       "datetime": pd.date_range(start="2020-07-01", end="2020-07-02", freq="15min"),
       "temperature": [25.0] * 97,
       "irradiance": [0.0] * 97,
       "wind_speed": [5.0] * 97
   })

In [24]:
# Set irradiance to follow a typical daily pattern
for i in range(97):
    hour = i // 4
    if 6 <= hour < 21:  # Daylight hours (6:00 - 21:00)
        # Simple bell curve for irradiance
        irradiance = 1000 * (1 - ((hour - 13.5) / 7.5) ** 2)
        if irradiance < 0:
            irradiance = 0
        weather_data.loc[i, "irradiance"] = irradiance

# # Load weather data
# env.load_weather_from_dataframe(
#     dataframe=weather_data,
#     datetime_column="datetime",
#     temperature_column="temperature",
#     irradiance_column="irradiance",
#     wind_speed_column="wind_speed"
# )

env.temp_data=weather_data['temperature']
env.pv_data=weather_data['irradiance']
env.wind_data=weather_data['wind_speed']

# Create a photovoltaic system
pv = vpplib.Photovoltaic(
    unit="kW",
    identifier="PV_1",
    environment=env,
    module_lib="SandiaMod",
    module="Canadian_Solar_CS5P_220M___2009_",
    inverter_lib="SandiaInverter",
    inverter="ABB__MICRO_0_25_I_OUTD_US_208__208V_",
    surface_tilt=30,
    surface_azimuth=180,
    modules_per_string=10,
    strings_per_inverter=2,
    latitude= 51.4,
    longitude= 6,
    temp_lib="sapm",             # <--- Add this
    temp_model="open_rack_glass_glass"  # <--- And this
)

# Create a battery
battery = vpplib.ElectricalEnergyStorage(
    unit="kW",
    identifier="Battery_1",
    environment=env,
    capacity=100,
    max_power=50,
    efficiency=0.95,
    self_discharge=0.001
)

# Create a user profile with a constant load of 2 kW
user = vpplib.UserProfile(timebase=15)
user.load_power_from_csv(
    filepath="load_profile.csv",
    datetime_column="datetime",
    power_column="power"
)

# If the load profile file doesn't exist, create a constant load
try:
    user.get_power("2020-07-01 00:00:00")
except:
    # Create a constant load of 2 kW
    load_data = pd.DataFrame({
        "datetime": pd.date_range(start="2020-07-01", end="2020-07-02", freq="15min"),
        "power": [2.0] * 97
    })
    user.load_power_from_dataframe(
        dataframe=load_data,
        datetime_column="datetime",
        power_column="power"
    )

# Create a virtual power plant
vpp = vpplib.VirtualPowerPlant(identifier="VPP_1")

# Add the photovoltaic system and battery to the virtual power plant
vpp.add_component(pv)
vpp.add_component(battery)

# Create an operator that maximizes self-consumption
class MaximizeSelfConsumption(vpplib.Operator):
    def operate(self, time):
        # Get the current power balance
        power_balance = self.vpp.get_power_balance(time)
        
        # Get the battery
        battery = self.vpp.get_component("Battery_1")
        
        # Get the user load
        user_load = user.get_power(time)
        
        # Calculate the net power (PV - load)
        pv_power = pv.get_power(time)
        net_power = pv_power - user_load
        
        # If there is excess power, charge the battery
        if net_power > 0:
            battery.charge(net_power, time)
        # If there is a power deficit, discharge the battery
        elif net_power < 0:
            battery.discharge(abs(net_power), time)

operator = MaximizeSelfConsumption(vpp=vpp)

TypeError: ElectricalEnergyStorage.__init__() got an unexpected keyword argument 'efficiency'

In [20]:
print(dir(env))


['_Environment__end_dt_target_tz', '_Environment__end_dt_utc', '_Environment__force_end_time', '_Environment__get_dwd_data', '_Environment__get_multi_index_for_windpowerlib', '_Environment__get_solar_parameter', '_Environment__get_solar_power_from_energy', '_Environment__get_station_pressure_from_reduced_pressure', '_Environment__process_mosmix_parameter', '_Environment__process_observation_parameter', '_Environment__resample_data', '_Environment__start_dt_target_tz', '_Environment__start_dt_utc', '_Environment__surpress_output_globally', '_Environment__use_timezone_aware_time_index', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'end', 'get_dwd_mean_quarter_hours', 'get_dwd_me

In [1]:
from open_mastr import Mastr

db = Mastr()
db.download()

Data will be written to the following database: sqlite:///C:\Users\ashka\.open-MaStR\data\sqlite\open-mastr.db
If you run into problems, try to delete the database and update the package by running 'pip install --upgrade open-mastr'

Download has started, this can take several minutes.The download bar is only a rough estimate.


C:\Users\ashka\.open-MaStR\data\xml_download\Gesamtdatenexport_20250704.zip:  87%|████████▋ | 15652/18000.0 [1:13:59<11:05,  3.53/s]                                                                                                          


Download is finished. It took 4440 seconds.
MaStR was successfully downloaded to C:\Users\ashka\.open-MaStR\data\xml_download.

Would you like to speed up the bulk download?
Try our new parallelized processing by setting os.environ['USE_RECOMMENDED_NUMBER_OF_PROCESSES'] = True or configure your own number of processes via os.environ['NUMBER_OF_PROCESSES'] = your_number

Starting bulk download and data cleansing...
Processing file 'AnlagenEegBiomasse.xml'...
Creating table 'biomass_eeg'...
Processing file 'AnlagenEegGeothermieGrubengasDruckentspannung.xml'...
Creating table 'gsgk_eeg'...
Processing file 'AnlagenEegSolar_1.xml'...
Creating table 'solar_eeg'...
Processing file 'AnlagenEegSpeicher_1.xml'...
Creating table 'storage_eeg'...
Processing file 'AnlagenEegWasser.xml'...
Creating table 'hydro_eeg'...
Processing file 'AnlagenEegWind.xml'...
Creating table 'wind_eeg'...
Processing file 'AnlagenGasSpeicher.xml'...
Creating table 'gas_storage'...
Processing file 'AnlagenKwk.xml'...
Cr

In [9]:
import os
os.environ['OUTPUT_PATH'] = r'D:\Projects\Kaggle projects\VPPlib_playground\.open-MaStR'
from open_mastr import Mastr
db = Mastr()


Data will be written to the following database: sqlite:///D:\Projects\Kaggle projects\VPPlib_playground\.open-MaStR\data\sqlite\open-mastr.db
If you run into problems, try to delete the database and update the package by running 'pip install --upgrade open-mastr'



In [11]:
import pandas as pd

# generate a list of all tables
df = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", con=db.engine)

# read the data of one table
table="wind_extended"
df = pd.read_sql(sql=table, con=db.engine)

df

,NameWindpark,Lage,Seelage,ClusterOstsee,ClusterNordsee,Hersteller,HerstellerId,Technologie,Typenbezeichnung,Nabenhoehe,...,PraequalifiziertFuerRegelenergie,GenMastrNummer,Netzbetreiberzuordnungen,ReserveartNachDemEnWG,DatumUeberfuehrungInReserve,Hausnummer_nv,Weic_nv,Kraftwerksnummer_nv,DatenQuelle,DatumDownload
0,Windpark Kreuzstein,None,None,None,None,ENERCON GmbH,None,Horizontalläufer,E-115,149.08,...,None,SGE961876398816,None,None,NaT,False,False,False,bulk,2025-07-04
1,Damsdorf,None,None,None,None,ENERCON GmbH,None,Horizontalläufer,E-115,135.40,...,None,SGE918200258391,None,None,NaT,False,False,False,bulk,2025-07-04
2,Windpark Kreuzstein,None,None,None,None,ENERCON GmbH,None,Horizontalläufer,E-115,149.08,...,None,SGE961876398816,None,None,NaT,False,False,False,bulk,2025-07-04
3,Windpark Kreuzstein,None,None,None,None,ENERCON GmbH,None,Horizontalläufer,E-115,149.08,...,None,SGE961876398816,None,None,NaT,False,False,False,bulk,2025-07-04
4,Windpark Heinsberg Pütt,None,None,None,None,Nordex SE,None,Horizontalläufer,N117,120.00,...,None,SGE950157614196,None,None,NaT,False,False,False,bulk,2025-07-04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39732,WP Harrenstätter Heide,None,None,None,None,None,None,Horizontalläufer,None,164.00,...,None,SGE931984592246,None,None,NaT,False,False,False,bulk,2025-07-04
39733,WP Harrenstätter Heide,None,None,None,None,None,None,Horizontalläufer,None,164.00,...,None,SGE931984592246,None,None,NaT,False,False,False,bulk,2025-07-04
39734,Manker-Protzen,None,None,None,None,Vestas Deutschland GmbH,None,Horizontalläufer,V162-6.0,169.00,...,None,SGE946024623286,None,None,NaT,False,False,False,bulk,2025-07-04
39735,Manker-Protzen,None,None,None,None,Vestas Deutschland GmbH,None,Horizontalläufer,V162-6.0,169.00,...,None,SGE946024623286,None,None,NaT,False,False,False,bulk,2025-07-04
